In [2]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TARGET_N = 5000
SAVE_EVERY = 50

BASE_SAVE_DIR = Path("encar_hyundai_all")
HTML_DIR = BASE_SAVE_DIR / "html"
OUTPUT_DIR = BASE_SAVE_DIR / "output"

BATCH_CSV = OUTPUT_DIR / "hyundai_detail_batch.csv"
ERROR_CSV = OUTPUT_DIR / "hyundai_detail_errors.csv"
FINAL_JSON = OUTPUT_DIR / "hyundai_detail_final.json"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C"
    "%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C"
    "%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C"
    "%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


def build_page_url(base_url, page_num):
    if "page=" in base_url:
        return re.sub(r"page=\d+", f"page={page_num}", base_url)
    sep = "&" if "?" in base_url else "?"
    return f"{base_url}{sep}page={page_num}"


# =========================
# Selenium
# =========================
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.2)
                except Exception:
                    pass
        except Exception:
            pass


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def collect_hyundai_links(driver, target_n=5000, max_pages=30, max_scrolls_per_page=20):
    all_links = []
    seen = set()
    no_new_page_count = 0

    for page_num in range(1, max_pages + 1):
        page_url = build_page_url(SEARCH_URL, page_num)
        print(f"\n[PAGE {page_num}] {page_url}")

        driver.get(page_url)
        wait_body(driver)
        time.sleep(2)
        close_popups(driver)

        last_count = 0
        stable_round = 0

        for scroll_idx in range(max_scrolls_per_page):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            page_links = collect_detail_links(driver)
            current_count = len(page_links)
            print(f"  [SCROLL {scroll_idx+1}] 현재 페이지 링크 수: {current_count}")

            if current_count == last_count:
                stable_round += 1
            else:
                stable_round = 0

            if stable_round >= 3:
                break

            last_count = current_count

        page_links = collect_detail_links(driver)

        new_count = 0
        for link in page_links:
            if link not in seen:
                seen.add(link)
                all_links.append(link)
                new_count += 1

        print(f"[PAGE {page_num}] 새로 추가된 링크 수: {new_count}")
        print(f"[TOTAL] 누적 링크 수: {len(all_links)}")

        if len(all_links) >= target_n:
            return all_links[:target_n]

        if new_count == 0:
            no_new_page_count += 1
        else:
            no_new_page_count = 0

        if no_new_page_count >= 2:
            print("[INFO] 연속 2페이지 동안 새 링크가 없어 중단")
            break

    return all_links[:target_n]


def fetch_and_save_detail_page(driver, url, html_path, txt_path=None, sleep_sec=2.0):
    driver.get(url)
    wait_body(driver)
    time.sleep(sleep_sec)
    close_popups(driver)
    time.sleep(0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
        txt_path.write_text(body_text, encoding="utf-8")

    return html


# =========================
# detail 파싱
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# =========================
# 저장 헬퍼
# =========================
def save_batch(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(BATCH_CSV, index=False, encoding="utf-8-sig")


def save_errors(errors):
    if not errors:
        return
    df = pd.DataFrame(errors)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")


# =========================
# 메인
# =========================
def main():
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)
    rows = []
    errors = []

    try:
        detail_links = collect_hyundai_links(driver, target_n=TARGET_N, max_pages=30)
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        seen_ids = set()

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)

            if not car_id:
                errors.append({
                    "idx": idx,
                    "상세링크": detail_url,
                    "error": "매물ID 추출 실패"
                })
                continue

            if car_id in seen_ids:
                continue
            seen_ids.add(car_id)

            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            try:
                car_dir = HTML_DIR / str(car_id)
                car_dir.mkdir(parents=True, exist_ok=True)

                detail_html = fetch_and_save_detail_page(
                    driver,
                    detail_url,
                    car_dir / "detail.html",
                    car_dir / "detail.txt"
                )

                row = parse_detail_file(detail_html, car_id_hint=car_id)
                row["상세링크"] = detail_url
                rows.append(row)

                print(f"[OK] {car_id} 완료 / 누적 {len(rows)}건")

                if len(rows) % SAVE_EVERY == 0:
                    print(f"[SAVE] {len(rows)}건 batch 저장")
                    save_batch(rows)
                    save_errors(errors)

                if len(rows) >= TARGET_N:
                    break

            except Exception as e:
                print(f"[ERROR] {car_id}: {e}")
                errors.append({
                    "idx": idx,
                    "매물ID": car_id,
                    "상세링크": detail_url,
                    "error": str(e)
                })

    finally:
        driver.quit()

    save_batch(rows)
    save_errors(errors)

    with open(FINAL_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {BATCH_CSV}")
    print(f"저장 완료: {ERROR_CSV}")
    print(f"저장 완료: {FINAL_JSON}")
    print(f"HTML 저장 폴더: {HTML_DIR.resolve()}")
    print(f"\n최종 수집 건수: {len(rows)}")
    print(f"에러 건수: {len(errors)}")

    if rows:
        df = pd.DataFrame(rows)
        show_cols = [
            "매물ID", "차량명", "현재가격_만원",
            "교환_개수", "판금_개수", "부식_여부",
            "보험이력건수", "사고강도점수", "사고종합_여부"
        ]
        show_cols = [c for c in show_cols if c in df.columns]
        print("\n[HEAD]")
        print(df[show_cols].head())


if __name__ == "__main__":
    main()


[PAGE 1] https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  [SCROLL 1] 현재 페이지 링크 수: 250
  [SCROLL 2] 현재 페이지 링크 수: 250
  [SCROLL 3] 현재 페이지 링크 수: 250
  [SCROLL 4] 현재 페이지 링크 수: 250
[PAGE 1] 새로 추가된 링크 수: 250
[TOTAL] 누적 링크 수: 250

[PAGE 2] https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  [SCROLL 1] 현재 페이지 링크 수: 200
  [SCROLL 2] 현재 페이지 링크 수: 200
  [SCROLL 3] 현재 페이지 링크 수: 200
  [SCROLL 4] 현재 페이지 링크 수: 200
[PAGE 2] 새로 추가된 링크 수: 200
[TOTAL] 누적 링크 수: 450

[PAGE 3] https://car.encar.com/list/car?page=3&search=%7B%22typ

In [3]:
import pandas as pd
from pathlib import Path

# 파일 경로
err_path = Path("encar_hyundai_all/output/hyundai_detail_errors.csv")

# 읽기
df_err = pd.read_csv(err_path)

print("행/열:", df_err.shape)
print("\n컬럼 목록:")
print(df_err.columns.tolist())

print("\n[HEAD]")
print(df_err.head())

# 에러 관련 컬럼 자동 탐색
candidate_cols = [c for c in df_err.columns if any(k in c.lower() for k in ["error", "err", "message", "msg", "reason"])]
print("\n에러 후보 컬럼:", candidate_cols)

# 가장 가능성 높은 에러 컬럼 선택
if len(candidate_cols) == 0:
    raise ValueError("에러 메시지로 보이는 컬럼을 찾지 못했습니다. 위 컬럼 목록을 보고 직접 지정해주세요.")

err_col = candidate_cols[0]
print(f"\n사용할 에러 컬럼: {err_col}")

# 문자열 정리
s = df_err[err_col].astype(str).fillna("")

def normalize_error(x: str) -> str:
    x_low = x.lower()

    if "403" in x_low:
        return "HTTP 403"
    elif "404" in x_low:
        return "HTTP 404"
    elif "429" in x_low:
        return "HTTP 429"
    elif "500" in x_low:
        return "HTTP 500"
    elif "timeout" in x_low or "timed out" in x_low:
        return "Timeout"
    elif "connection" in x_low:
        return "Connection Error"
    elif "json" in x_low:
        return "JSON Parse Error"
    elif "selector" in x_low or "not found" in x_low:
        return "Selector/Element Missing"
    elif "nonetype" in x_low:
        return "NoneType Error"
    elif "indexerror" in x_low:
        return "Index Error"
    elif "keyerror" in x_low:
        return "Key Error"
    elif "valueerror" in x_low:
        return "Value Error"
    elif "attributeerror" in x_low:
        return "Attribute Error"
    elif "webdriver" in x_low or "chrome" in x_low:
        return "Browser/Driver Error"
    elif "보험" in x_low:
        return "보험 섹션 문제"
    elif "성능" in x_low or "점검" in x_low:
        return "성능점검 섹션 문제"
    else:
        return "기타"

df_err["error_group"] = s.apply(normalize_error)

print("\n[에러 그룹별 건수]")
print(df_err["error_group"].value_counts(dropna=False))

print("\n[에러 원문 상위 20개]")
print(df_err[err_col].value_counts(dropna=False).head(20))

print("\n[그룹별 샘플 3개]")
for g in df_err["error_group"].dropna().unique():
    print(f"\n### {g}")
    print(df_err.loc[df_err["error_group"] == g, [err_col]].head(3).to_string(index=False))

행/열: (722, 4)

컬럼 목록:
['idx', '매물ID', '상세링크', 'error']

[HEAD]
   idx      매물ID                                               상세링크  \
0  236  40945676  https://fem.encar.com/cars/detail/40945676?adv...   
1  237  41660601  https://fem.encar.com/cars/detail/41660601?adv...   
2  238  41662129  https://fem.encar.com/cars/detail/41662129?adv...   
3  239  41653279  https://fem.encar.com/cars/detail/41653279?adv...   
4  240  41653283  https://fem.encar.com/cars/detail/41653283?adv...   

                                               error  
0  HTTPConnectionPool(host='localhost', port=6166...  
1  HTTPConnectionPool(host='localhost', port=6166...  
2  HTTPConnectionPool(host='localhost', port=6166...  
3  HTTPConnectionPool(host='localhost', port=6166...  
4  HTTPConnectionPool(host='localhost', port=6166...  

에러 후보 컬럼: ['error']

사용할 에러 컬럼: error

[에러 그룹별 건수]
error_group
Browser/Driver Error    716
Timeout                   6
Name: count, dtype: int64

[에러 원문 상위 20개]
error
Message: tab

In [4]:
from pathlib import Path

html_dir = Path("/Users/chanho0123/hipython/버뮤다_프로젝트_2/encar_hyundai_all/html")

html_files = sorted(html_dir.glob("*.html"))

print("HTML 파일 수:", len(html_files))

for fp in html_files[:5]:
    text = fp.read_text(encoding="utf-8", errors="ignore")
    print("\n" + "="*80)
    print("파일명:", fp.name)
    print("길이:", len(text))

    # 앞부분만 보기
    preview = text[:800].replace("\n", " ")
    print("미리보기:", preview[:500])

    # 간단한 판별
    text_low = text.lower()
    flags = {
        "encar 포함": "encar" in text_low,
        "차량명/vehicle 관련": ("차량" in text) or ("vehicle" in text_low),
        "보험 포함": "보험" in text,
        "성능점검 포함": ("성능점검" in text) or ("점검" in text),
        "script 과다": text_low.count("<script") > 20,
        "너무 짧음": len(text) < 5000,
    }
    print("판별:", flags)

HTML 파일 수: 0


In [5]:
import re
import time
import json
import random
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TARGET_N = 5000
SAVE_EVERY = 50

MAX_PAGES = 30
MAX_SCROLLS_PER_PAGE = 20

DETAIL_SLEEP_SEC = 1.2
RETRY_PER_CAR = 2
RESTART_INTERVAL = 40          # 30~50 추천
PAGE_LOAD_TIMEOUT = 25
WAIT_BODY_SEC = 8

BASE_SAVE_DIR = Path("encar_hyundai_all")
HTML_DIR = BASE_SAVE_DIR / "html"
OUTPUT_DIR = BASE_SAVE_DIR / "output"

BATCH_CSV = OUTPUT_DIR / "hyundai_detail_batch.csv"
ERROR_CSV = OUTPUT_DIR / "hyundai_detail_errors.csv"
FINAL_JSON = OUTPUT_DIR / "hyundai_detail_final.json"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C"
    "%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C"
    "%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C"
    "%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            yy = int(m.group(1))
            return 2000 + yy if yy < 30 else 1900 + yy
    return None


def build_page_url(base_url, page_num):
    if "page=" in base_url:
        return re.sub(r"page=\d+", f"page={page_num}", base_url)
    sep = "&" if "?" in base_url else "?"
    return f"{base_url}{sep}page={page_num}"


def short_sleep(a=0.5, b=1.0):
    time.sleep(random.uniform(a, b))


def is_tab_crash_error(e: Exception) -> bool:
    s = str(e).lower()
    return "tab crashed" in s or "session deleted because of page crash" in s


def is_timeout_error(e: Exception) -> bool:
    s = str(e).lower()
    return "timed out" in s or "timeout" in s


# =========================
# Selenium
# =========================
def setup_driver(headless=True):
    options = Options()

    # 안정화 옵션
    if headless:
        options.add_argument("--headless=new")

    options.page_load_strategy = "eager"  # 완전 로딩까지 안 기다림
    options.add_argument("--window-size=1400,2200")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-background-networking")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-features=Translate,BackForwardCache,AcceptCHFrame")
    options.add_argument("--hide-scrollbars")
    options.add_argument("--mute-audio")
    options.add_argument("--log-level=3")
    options.add_argument("--remote-allow-origins=*")

    # 메모리 절약: 이미지 끄기
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.geolocation": 2,
    }
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option("excludeSwitches", ["enable-automation", "enable-logging"])
    options.add_experimental_option("useAutomationExtension", False)

    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.implicitly_wait(2)

    try:
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {
                "source": """
                    Object.defineProperty(navigator, 'webdriver', {
                        get: () => undefined
                    });
                """
            },
        )
    except Exception:
        pass

    return driver


def restart_driver(driver, headless=True):
    try:
        if driver is not None:
            driver.quit()
    except Exception:
        pass
    short_sleep(0.5, 1.0)
    return setup_driver(headless=headless)


def wait_body(driver, sec=WAIT_BODY_SEC):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def safe_get(driver, url, wait_sec=WAIT_BODY_SEC):
    driver.get(url)
    wait_body(driver, sec=wait_sec)
    return True


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons[:3]:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.15)
                except Exception:
                    pass
        except Exception:
            pass


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def collect_hyundai_links(driver, target_n=5000, max_pages=30, max_scrolls_per_page=20, headless=True):
    all_links = []
    seen = set()
    no_new_page_count = 0

    for page_num in range(1, max_pages + 1):
        page_url = build_page_url(SEARCH_URL, page_num)
        print(f"\n[PAGE {page_num}] {page_url}")

        try:
            safe_get(driver, page_url)
            short_sleep(1.0, 1.8)
            close_popups(driver)
        except Exception as e:
            print(f"[WARN] 목록 페이지 로드 실패, driver 재시작: {e}")
            driver = restart_driver(driver, headless=headless)
            safe_get(driver, page_url)
            short_sleep(1.0, 1.8)

        last_count = 0
        stable_round = 0

        for scroll_idx in range(max_scrolls_per_page):
            try:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            except Exception:
                pass

            short_sleep(1.0, 1.7)
            page_links = collect_detail_links(driver)
            current_count = len(page_links)
            print(f"  [SCROLL {scroll_idx+1}] 현재 페이지 링크 수: {current_count}")

            if current_count == last_count:
                stable_round += 1
            else:
                stable_round = 0

            if stable_round >= 3:
                break

            last_count = current_count

        page_links = collect_detail_links(driver)

        new_count = 0
        for link in page_links:
            if link not in seen:
                seen.add(link)
                all_links.append(link)
                new_count += 1

        print(f"[PAGE {page_num}] 새로 추가된 링크 수: {new_count}")
        print(f"[TOTAL] 누적 링크 수: {len(all_links)}")

        if len(all_links) >= target_n:
            return all_links[:target_n], driver

        if new_count == 0:
            no_new_page_count += 1
        else:
            no_new_page_count = 0

        if no_new_page_count >= 2:
            print("[INFO] 연속 2페이지 동안 새 링크가 없어 중단")
            break

    return all_links[:target_n], driver


def fetch_and_save_detail_page(driver, url, html_path, txt_path=None, sleep_sec=1.2):
    safe_get(driver, url)
    short_sleep(sleep_sec, sleep_sec + 0.7)
    close_popups(driver)
    short_sleep(0.2, 0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        try:
            body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
            txt_path.write_text(body_text, encoding="utf-8")
        except Exception:
            txt_path.write_text("", encoding="utf-8")

    return html


# =========================
# detail 파싱
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(
            emb.get("yearMonth"),
            emb.get("formYear"),
            dom.get("연식_원문_dom") or meta.get("연식_메타")
        ),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# =========================
# 저장 헬퍼
# =========================
def save_batch(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(BATCH_CSV, index=False, encoding="utf-8-sig")


def save_errors(errors):
    if not errors:
        return
    df = pd.DataFrame(errors)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")


# =========================
# 메인
# =========================
def main():
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=True)
    rows = []
    errors = []

    try:
        detail_links, driver = collect_hyundai_links(
            driver,
            target_n=TARGET_N,
            max_pages=MAX_PAGES,
            max_scrolls_per_page=MAX_SCROLLS_PER_PAGE,
            headless=True,
        )
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        seen_ids = set()
        success_count = 0

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)

            if not car_id:
                errors.append({
                    "idx": idx,
                    "상세링크": detail_url,
                    "error": "매물ID 추출 실패"
                })
                continue

            if car_id in seen_ids:
                continue
            seen_ids.add(car_id)

            # 일정 개수마다 브라우저 재시작
            if success_count > 0 and success_count % RESTART_INTERVAL == 0:
                print(f"[INFO] {success_count}건 처리 후 driver 재시작")
                driver = restart_driver(driver, headless=True)

            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            ok = False
            last_error = None

            for attempt in range(1, RETRY_PER_CAR + 1):
                try:
                    car_dir = HTML_DIR / str(car_id)
                    car_dir.mkdir(parents=True, exist_ok=True)

                    detail_html = fetch_and_save_detail_page(
                        driver,
                        detail_url,
                        car_dir / "detail.html",
                        car_dir / "detail.txt",
                        sleep_sec=DETAIL_SLEEP_SEC
                    )

                    row = parse_detail_file(detail_html, car_id_hint=car_id)
                    row["상세링크"] = detail_url
                    rows.append(row)

                    success_count += 1
                    ok = True

                    print(f"[OK] {car_id} 완료 / 누적 {len(rows)}건")
                    break

                except Exception as e:
                    last_error = str(e)
                    print(f"[WARN] {car_id} attempt={attempt} 실패: {e}")

                    if is_tab_crash_error(e) or is_timeout_error(e):
                        print("[INFO] 브라우저 재시작 후 재시도")
                        driver = restart_driver(driver, headless=True)
                        short_sleep(0.8, 1.5)
                    else:
                        short_sleep(0.5, 1.0)

            if not ok:
                errors.append({
                    "idx": idx,
                    "매물ID": car_id,
                    "상세링크": detail_url,
                    "error": last_error if last_error else "unknown error"
                })
                print(f"[ERROR] {car_id}: 최종 실패")

            if len(rows) % SAVE_EVERY == 0 and len(rows) > 0:
                print(f"[SAVE] {len(rows)}건 batch 저장")
                save_batch(rows)
                save_errors(errors)

            if len(rows) >= TARGET_N:
                break

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    save_batch(rows)
    save_errors(errors)

    with open(FINAL_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {BATCH_CSV}")
    print(f"저장 완료: {ERROR_CSV}")
    print(f"저장 완료: {FINAL_JSON}")
    print(f"HTML 저장 폴더: {HTML_DIR.resolve()}")
    print(f"\n최종 수집 건수: {len(rows)}")
    print(f"에러 건수: {len(errors)}")

    if rows:
        df = pd.DataFrame(rows)
        show_cols = [
            "매물ID", "차량명", "현재가격_만원",
            "교환_개수", "판금_개수", "부식_여부",
            "보험이력건수", "사고강도점수", "사고종합_여부"
        ]
        show_cols = [c for c in show_cols if c in df.columns]
        print("\n[HEAD]")
        print(df[show_cols].head())


if __name__ == "__main__":
    main()


[PAGE 1] https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  [SCROLL 1] 현재 페이지 링크 수: 250
  [SCROLL 2] 현재 페이지 링크 수: 250
  [SCROLL 3] 현재 페이지 링크 수: 250
  [SCROLL 4] 현재 페이지 링크 수: 250
[PAGE 1] 새로 추가된 링크 수: 250
[TOTAL] 누적 링크 수: 250

[PAGE 2] https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  [SCROLL 1] 현재 페이지 링크 수: 200
  [SCROLL 2] 현재 페이지 링크 수: 200
  [SCROLL 3] 현재 페이지 링크 수: 200
  [SCROLL 4] 현재 페이지 링크 수: 200
[PAGE 2] 새로 추가된 링크 수: 200
[TOTAL] 누적 링크 수: 450

[PAGE 3] https://car.encar.com/list/car?page=3&search=%7B%22typ

In [8]:
import re
import json
import time
import math
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================================================
# 설정
# =========================================================
TOTAL_TARGET = 30000
SAVE_EVERY = 200

MAX_PAGES_PER_BRAND = 200
MAX_SCROLLS_PER_PAGE = 20
LIST_RESTART_INTERVAL = 8

REQUEST_SLEEP_RANGE = (0.6, 1.1)
DETAIL_SLEEP_RANGE = (0.7, 1.3)

PAGE_LOAD_TIMEOUT = 25
WAIT_BODY_SEC = 8
TIMEOUT = 20

BASE_DIR = Path("encar_domestic_hybrid_30000")
OUTPUT_DIR = BASE_DIR / "output"
HTML_DIR = BASE_DIR / "html"

RAW_CSV = OUTPUT_DIR / "domestic_raw_30000.csv"
MODEL_CSV = OUTPUT_DIR / "domestic_model_30000.csv"
ERROR_CSV = OUTPUT_DIR / "domestic_errors_30000.csv"
LINKS_CSV = OUTPUT_DIR / "domestic_links_30000.csv"
RAW_JSON = OUTPUT_DIR / "domestic_raw_30000.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)

BRAND_URLS = {
    "현대": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D",
    "제네시스": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%EC%A0%9C%EB%84%A4%EC%8B%9C%EC%8A%A4.))%22%2C%22title%22%3A%22%EC%A0%9C%EB%84%A4%EC%8B%9C%EC%8A%A4%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%2C%22cursor%22%3A%22%22%7D",
    "기아": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%EA%B8%B0%EC%95%84.))%22%2C%22title%22%3A%22%EA%B8%B0%EC%95%84%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%2C%22cursor%22%3A%22%22%7D",
    "쉐보레": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%EC%89%90%EB%B3%B4%EB%A0%88(GM%EB%8C%80%EC%9A%B0_).))%22%2C%22title%22%3A%22%EC%89%90%EB%B3%B4%EB%A0%88(GM%EB%8C%80%EC%9A%B0)%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%2C%22cursor%22%3A%22%22%7D",
    "르노": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%EB%A5%B4%EB%85%B8%EC%BD%94%EB%A6%AC%EC%95%84(%EC%82%BC%EC%84%B1_).))%22%2C%22title%22%3A%22%EB%A5%B4%EB%85%B8%EC%BD%94%EB%A6%AC%EC%95%84(%EC%82%BC%EC%84%B1)%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%2C%22cursor%22%3A%22%22%7D",
    "KGM": "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.KG%EB%AA%A8%EB%B9%8C%EB%A6%AC%ED%8B%B0(%EC%8C%8D%EC%9A%A9_).))%22%2C%22title%22%3A%22KG%EB%AA%A8%EB%B9%8C%EB%A6%AC%ED%8B%B0(%EC%8C%8D%EC%9A%A9)%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%2C%22cursor%22%3A%22%22%7D",
}

BRAND_TARGETS = {
    "현대": 10400,
    "제네시스": 2900,
    "기아": 11000,
    "쉐보레": 2100,
    "르노": 1500,
    "KGM": 2100,
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://car.encar.com/",
    "Connection": "keep-alive",
}


# =========================================================
# 유틸
# =========================================================
def short_sleep(rng: Tuple[float, float]) -> None:
    time.sleep(random.uniform(*rng))


def normalize_text(text: Optional[str]) -> str:
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def to_int(val) -> Optional[int]:
    if val is None:
        return None
    s = re.sub(r"[^\d\-]", "", str(val))
    return int(s) if s not in {"", "-"} else None


def parse_int_from_text(text: Optional[str]) -> Optional[int]:
    if not text:
        return None
    m = re.search(r"([\d,]+)", str(text))
    return int(m.group(1).replace(",", "")) if m else None


def extract_car_id(url: str) -> Optional[str]:
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_page_url(base_url: str, page_num: int) -> str:
    if "page=" in base_url:
        return re.sub(r"page=\d+", f"page={page_num}", base_url)
    sep = "&" if "?" in base_url else "?"
    return f"{base_url}{sep}page={page_num}"


def safe_json_loads(text: Optional[str], default=None):
    if default is None:
        default = {}
    if not text:
        return default
    try:
        return json.loads(text)
    except Exception:
        return default


def find_first(patterns: List[str], text: str, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(str(year_month)) >= 4:
        try:
            return int(str(year_month)[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", str(fallback_text))
        if m:
            yy = int(m.group(1))
            return 2000 + yy if yy < 30 else 1900 + yy
    return None


# =========================================================
# 세션 / 기존 데이터
# =========================================================
def create_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def load_existing_ids() -> set:
    if RAW_CSV.exists():
        df = pd.read_csv(RAW_CSV)
        if "매물ID" in df.columns:
            return set(df["매물ID"].astype(str))
    return set()


def append_csv(path: Path, rows: List[Dict]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")


# =========================================================
# Selenium 목록 수집
# =========================================================
def setup_driver(headless=True):
    options = Options()

    if headless:
        options.add_argument("--headless=new")

    options.page_load_strategy = "eager"
    options.add_argument("--window-size=1400,2200")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-background-networking")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-features=Translate,BackForwardCache,AcceptCHFrame")
    options.add_argument("--hide-scrollbars")
    options.add_argument("--mute-audio")
    options.add_argument("--log-level=3")
    options.add_argument("--remote-allow-origins=*")

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.geolocation": 2,
    }
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option("excludeSwitches", ["enable-automation", "enable-logging"])
    options.add_experimental_option("useAutomationExtension", False)

    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.implicitly_wait(2)
    return driver


def restart_driver(driver, headless=True):
    try:
        if driver is not None:
            driver.quit()
    except Exception:
        pass
    short_sleep((0.5, 1.0))
    return setup_driver(headless=headless)


def wait_body(driver, sec=WAIT_BODY_SEC):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons[:3]:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.15)
                except Exception:
                    pass
        except Exception:
            pass


def collect_detail_links_from_current_page(driver) -> List[str]:
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def collect_detail_links_for_brand_selenium(
    brand_name: str,
    list_url: str,
    brand_target: int,
    existing_ids: set,
) -> List[str]:
    print(f"\n===== [{brand_name}] 목록 수집 시작 =====")
    driver = setup_driver(headless=True)
    all_links = []
    seen = set()
    no_new_page_count = 0

    try:
        for page_num in range(1, MAX_PAGES_PER_BRAND + 1):
            if page_num > 1 and page_num % LIST_RESTART_INTERVAL == 0:
                print(f"[{brand_name}] 목록 driver 재시작")
                driver = restart_driver(driver, headless=True)

            page_url = build_page_url(list_url, page_num)
            print(f"[{brand_name}] PAGE {page_num}")

            try:
                driver.get(page_url)
                wait_body(driver)
                short_sleep((1.0, 1.8))
                close_popups(driver)
            except Exception as e:
                print(f"[{brand_name}] PAGE {page_num} 로드 실패: {e}")
                driver = restart_driver(driver, headless=True)
                try:
                    driver.get(page_url)
                    wait_body(driver)
                    short_sleep((1.0, 1.8))
                except Exception as e2:
                    print(f"[{brand_name}] PAGE {page_num} 재시도 실패: {e2}")
                    break

            last_count = 0
            stable_round = 0

            for scroll_idx in range(MAX_SCROLLS_PER_PAGE):
                try:
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                except Exception:
                    pass
                short_sleep((0.8, 1.5))

                page_links = collect_detail_links_from_current_page(driver)
                current_count = len(page_links)

                if current_count == last_count:
                    stable_round += 1
                else:
                    stable_round = 0

                if stable_round >= 3:
                    break

                last_count = current_count

            page_links = collect_detail_links_from_current_page(driver)

            page_new = 0
            for link in page_links:
                car_id = extract_car_id(link)
                if not car_id:
                    continue
                if car_id in existing_ids:
                    continue
                if link in seen:
                    continue

                seen.add(link)
                all_links.append(link)
                page_new += 1

                if len(all_links) >= brand_target:
                    print(f"[{brand_name}] 목표 링크 도달: {len(all_links)}")
                    return all_links

            print(f"[{brand_name}] PAGE {page_num} 신규 링크: {page_new} / 누적: {len(all_links)}")

            if page_new == 0:
                no_new_page_count += 1
            else:
                no_new_page_count = 0

            if no_new_page_count >= 3:
                print(f"[{brand_name}] 연속 3페이지 신규 링크 없음, 중단")
                break

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    return all_links


# =========================================================
# requests 상세 수집 / 파싱
# =========================================================
def fetch_html(session: requests.Session, url: str, timeout: int = TIMEOUT) -> str:
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def parse_meta_description(soup: BeautifulSoup) -> Dict:
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup: BeautifulSoup) -> Dict:
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    for dl in soup.find_all("dl"):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        if not dts or not dds:
            continue
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k and not result["연식_원문_dom"]:
                result["연식_원문_dom"] = v
            elif "주행거리" in k and not result["주행거리_원문_dom"]:
                result["주행거리_원문_dom"] = v
            elif "연료" in k and not result["연료_dom"]:
                result["연료_dom"] = v
            elif "차량번호" in k and not result["차량번호_dom"]:
                result["차량번호_dom"] = v

    lis = soup.find_all("li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호") and not result["등록번호_dom"]:
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수") and result["조회수_dom"] is None:
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜") and result["찜수_dom"] is None:
            result["찜수_dom"] = parse_int_from_text(txt)

    return result


def parse_detail_embedded(html: str) -> Dict:
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = safe_json_loads(standard_raw, [])
    result["option_choice_codes"] = safe_json_loads(choice_raw, [])
    result["option_tuning_codes"] = safe_json_loads(tuning_raw, [])

    if etc_raw and str(etc_raw).startswith("["):
        result["option_etc_values"] = safe_json_loads(etc_raw, [])
    elif etc_raw:
        result["option_etc_values"] = [str(etc_raw).strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_insurance_and_damage(soup: BeautifulSoup) -> Dict:
    result = {
        "보험이력건수": None,
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    all_text = normalize_text(soup.get_text(" ", strip=True))

    m = re.search(r"보험이력\s*(\d+)건", all_text)
    if m:
        result["보험이력건수"] = int(m.group(1))

    items = []
    for li in soup.find_all("li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if not txt:
            continue

        if "교환" in txt or "판금" in txt or "부식" in txt:
            items.append(txt)

        if "교환" in txt and result["교환_개수"] is None:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        if "판금" in txt and result["판금_개수"] is None:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        if "부식" in txt and result["부식_여부"] is None:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row: Dict) -> Dict:
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = [v for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]] if v is not None]
    row["사고종합_여부"] = None if not signals else int(any(signals))
    row["중대사고_추정"] = None if row["사고강도점수"] is None else int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)
    row["부식_위험"] = corrosion
    return row


def parse_detail_html(html: str, brand_name: str, detail_url: str, car_id_hint: Optional[str]) -> Dict:
    soup = BeautifulSoup(html, "lxml")
    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(html)
    dmg = parse_insurance_and_damage(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")
    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "브랜드": brand_name,
        "매물ID": str(emb.get("vehicleId") or car_id_hint),
        "상세링크": detail_url,
        "제조사": emb.get("manufacturerName") or brand_name,
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "딜러명": emb.get("dealerName"),
        "상사명": emb.get("firmName"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
    }

    row.update(dmg)
    row = derive_accident_features(row)

    current_year = 2026
    row["차량연령"] = None if row["연식"] is None else max(current_year - row["연식"], 0)
    row["주행거리_로그"] = None if row["주행거리_km"] in [None, 0] else math.log1p(row["주행거리_km"])
    row["가격로그"] = None if row["현재가격_만원"] in [None, 0] else math.log1p(row["현재가격_만원"])

    return row


# =========================================================
# 모델용 테이블 생성
# =========================================================
def build_model_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    keep_cols = [
        "매물ID", "브랜드", "제조사", "모델", "등급명", "세부트림", "차량명",
        "현재가격_만원", "가격로그",
        "연식", "차량연령",
        "주행거리_km", "주행거리_로그",
        "연료", "변속기", "색상", "지역", "차급", "배기량_cc", "좌석수",
        "압류건수", "저당건수",
        "보험이력건수", "보험이력_여부",
        "교환_개수", "판금_개수", "부식_여부",
        "교환_여부", "판금_여부", "외판수리_여부",
        "사고강도점수", "사고종합_여부", "중대사고_추정", "부식_위험",
        "옵션_기본코드개수", "옵션_선택코드개수", "옵션_튜닝코드개수", "옵션_기타개수",
        "엔카진단여부값", "엔카믿고값",
        "조회수", "찜수",
    ]
    keep_cols = [c for c in keep_cols if c in df_raw.columns]
    df = df_raw[keep_cols].copy()

    if "매물ID" in df.columns:
        df = df.drop_duplicates(subset=["매물ID"], keep="first")

    if "현재가격_만원" in df.columns:
        df = df[df["현재가격_만원"].notna()]

    return df


# =========================================================
# 메인
# =========================================================
def main():
    session = create_session()
    existing_ids = load_existing_ids()

    print(f"[INFO] 기존 매물ID 수: {len(existing_ids)}")

    # 1. 목록 링크 수집
    all_link_rows: List[Dict] = []
    for brand_name, brand_url in BRAND_URLS.items():
        brand_target = BRAND_TARGETS.get(brand_name, 0)

        brand_links = collect_detail_links_for_brand_selenium(
            brand_name=brand_name,
            list_url=brand_url,
            brand_target=brand_target,
            existing_ids=existing_ids,
        )

        for u in brand_links:
            car_id = extract_car_id(u)
            all_link_rows.append({
                "브랜드": brand_name,
                "매물ID": car_id,
                "상세링크": u,
            })

    if all_link_rows:
        df_links = pd.DataFrame(all_link_rows).drop_duplicates(subset=["매물ID"], keep="first")
        df_links.to_csv(LINKS_CSV, index=False, encoding="utf-8-sig")
    else:
        df_links = pd.DataFrame(columns=["브랜드", "매물ID", "상세링크"])

    print(f"\n[INFO] 상세 수집 대상 총 링크 수: {len(df_links)}")

    # 2. 상세 수집
    rows: List[Dict] = []
    errors: List[Dict] = []
    seen_ids = set(existing_ids)
    total_success = 0

    for idx, rec in enumerate(df_links.to_dict(orient="records"), start=1):
        if total_success >= TOTAL_TARGET:
            break

        brand_name = rec["브랜드"]
        detail_url = rec["상세링크"]
        car_id = str(rec["매물ID"]) if rec["매물ID"] is not None else extract_car_id(detail_url)

        if not car_id or car_id in seen_ids:
            continue

        try:
            html = fetch_html(session, detail_url)
            short_sleep(DETAIL_SLEEP_RANGE)

            car_dir = HTML_DIR / brand_name / str(car_id)
            car_dir.mkdir(parents=True, exist_ok=True)
            (car_dir / "detail.html").write_text(html, encoding="utf-8")

            row = parse_detail_html(
                html=html,
                brand_name=brand_name,
                detail_url=detail_url,
                car_id_hint=car_id,
            )

            rows.append(row)
            seen_ids.add(car_id)
            total_success += 1

            print(f"[OK] {idx}/{len(df_links)} {brand_name} {car_id} / 누적 {total_success}")

        except Exception as e:
            errors.append({
                "idx": idx,
                "브랜드": brand_name,
                "매물ID": car_id,
                "상세링크": detail_url,
                "error": str(e),
            })
            print(f"[ERROR] {idx}/{len(df_links)} {brand_name} {car_id}: {e}")

        if len(rows) >= SAVE_EVERY:
            append_csv(RAW_CSV, rows)
            append_csv(ERROR_CSV, errors)
            rows = []
            errors = []

    append_csv(RAW_CSV, rows)
    append_csv(ERROR_CSV, errors)

    # 3. 최종 정리
    if RAW_CSV.exists():
        df_raw = pd.read_csv(RAW_CSV)
        if "매물ID" in df_raw.columns:
            df_raw = df_raw.drop_duplicates(subset=["매물ID"], keep="first")
        df_raw.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")

        df_model = build_model_df(df_raw)
        df_model.to_csv(MODEL_CSV, index=False, encoding="utf-8-sig")

        with open(RAW_JSON, "w", encoding="utf-8") as f:
            json.dump(df_raw.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

        print("\n===== 수집 완료 =====")
        print(f"RAW rows: {len(df_raw)}")
        print(f"MODEL rows: {len(df_model)}")
        print(f"RAW CSV: {RAW_CSV}")
        print(f"MODEL CSV: {MODEL_CSV}")
        print(f"ERROR CSV: {ERROR_CSV}")
        print(f"LINKS CSV: {LINKS_CSV}")
        print(f"RAW JSON: {RAW_JSON}")

        if "브랜드" in df_raw.columns:
            print("\n[브랜드별 건수]")
            print(df_raw["브랜드"].value_counts(dropna=False))


if __name__ == "__main__":
    main()

[INFO] 기존 매물ID 수: 0

===== [현대] 목록 수집 시작 =====
[현대] PAGE 1
[현대] PAGE 1 신규 링크: 250 / 누적: 250
[현대] PAGE 2
[현대] PAGE 2 신규 링크: 200 / 누적: 450
[현대] PAGE 3
[현대] PAGE 3 신규 링크: 200 / 누적: 650
[현대] PAGE 4
[현대] PAGE 4 신규 링크: 200 / 누적: 850
[현대] PAGE 5
[현대] PAGE 5 신규 링크: 200 / 누적: 1050
[현대] PAGE 6
[현대] PAGE 6 신규 링크: 200 / 누적: 1250
[현대] PAGE 7
[현대] PAGE 7 신규 링크: 200 / 누적: 1450
[현대] 목록 driver 재시작
[현대] PAGE 8
[현대] PAGE 8 신규 링크: 200 / 누적: 1650
[현대] PAGE 9
[현대] PAGE 9 신규 링크: 200 / 누적: 1850
[현대] PAGE 10
[현대] PAGE 10 신규 링크: 200 / 누적: 2050
[현대] PAGE 11
[현대] PAGE 11 신규 링크: 200 / 누적: 2250
[현대] PAGE 12
[현대] PAGE 12 신규 링크: 200 / 누적: 2450
[현대] PAGE 13
[현대] PAGE 13 신규 링크: 200 / 누적: 2650
[현대] PAGE 14
[현대] PAGE 14 신규 링크: 200 / 누적: 2850
[현대] PAGE 15
[현대] PAGE 15 신규 링크: 200 / 누적: 3050
[현대] 목록 driver 재시작
[현대] PAGE 16
[현대] PAGE 16 신규 링크: 200 / 누적: 3250
[현대] PAGE 17
[현대] PAGE 17 신규 링크: 200 / 누적: 3450
[현대] PAGE 18
[현대] PAGE 18 신규 링크: 200 / 누적: 3650
[현대] PAGE 19
[현대] PAGE 19 신규 링크: 200 / 누적: 3850
[현대] PAGE 20
[현대] PAGE 20

In [9]:
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


# =========================
# 경로 설정
# =========================
BASE_DIR = Path("encar_domestic_hybrid_30000")
RAW_CSV = BASE_DIR / "output" / "domestic_raw_30000.csv"
PATCHED_CSV = BASE_DIR / "output" / "domestic_raw_30000_accident_patched.csv"
HTML_DIR = BASE_DIR / "html"


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", str(text))
    return int(m.group(1).replace(",", "")) if m else None


# =========================
# 사고/보험 재파싱
# =========================
def parse_accident_from_html(html: str):
    soup = BeautifulSoup(html, "lxml")
    full_text = normalize_text(soup.get_text("\n", strip=True))
    lines = [normalize_text(x) for x in full_text.split("\n")]
    lines = [x for x in lines if x]

    result = {
        "보험이력건수_patch": None,
        "교환_개수_patch": None,
        "판금_개수_patch": None,
        "부식_여부_patch": None,
        "성능기록부_raw_patch": None,
    }

    evidence_lines = []

    # -------------------------
    # 보험이력
    # -------------------------
    insurance_patterns = [
        r"보험이력\s*(\d+)\s*건",
        r"보험\s*이력\s*(\d+)\s*건",
    ]

    for pat in insurance_patterns:
        m = re.search(pat, full_text)
        if m:
            result["보험이력건수_patch"] = int(m.group(1))
            break

    # -------------------------
    # 교환 / 판금 / 부식
    # 결과 문구만 인정
    # -------------------------
    for line in lines:
        # 증거 후보 저장
        if any(k in line for k in ["교환", "판금", "부식", "보험이력", "성능점검", "성능기록부"]):
            evidence_lines.append(line)

        # ---- 교환 ----
        if result["교환_개수_patch"] is None and "교환" in line:
            if re.search(r"교환\s*없음", line):
                result["교환_개수_patch"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", line)
                if m:
                    result["교환_개수_patch"] = int(m.group(1))

        # ---- 판금 ----
        if result["판금_개수_patch"] is None and "판금" in line:
            if re.search(r"판금\s*없음", line):
                result["판금_개수_patch"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", line)
                if m:
                    result["판금_개수_patch"] = int(m.group(1))

        # ---- 부식 ----
        # "부식 상태를 확인합니다" 같은 설명문은 제외
        if result["부식_여부_patch"] is None and "부식" in line:
            if re.search(r"부식\s*없음", line):
                result["부식_여부_patch"] = 0
            elif re.search(r"부식\s*(있음|발견|확인|존재)", line):
                result["부식_여부_patch"] = 1
            elif "부식 상태" in line and any(k in line for k in ["확인", "점검", "하체"]):
                # 안내성 문구는 판단 보류
                pass

    # -------------------------
    # 혹시 한 줄이 아니라 붙어 있는 경우 대비
    # -------------------------
    if result["교환_개수_patch"] is None:
        if re.search(r"교환\s*없음", full_text):
            result["교환_개수_patch"] = 0
        else:
            m = re.search(r"교환\s*(\d+)", full_text)
            if m:
                result["교환_개수_patch"] = int(m.group(1))

    if result["판금_개수_patch"] is None:
        if re.search(r"판금\s*없음", full_text):
            result["판금_개수_patch"] = 0
        else:
            m = re.search(r"판금\s*(\d+)", full_text)
            if m:
                result["판금_개수_patch"] = int(m.group(1))

    if result["부식_여부_patch"] is None:
        if re.search(r"부식\s*없음", full_text):
            result["부식_여부_patch"] = 0
        elif re.search(r"부식\s*(있음|발견|확인|존재)", full_text):
            # 설명문구 오탐 방지
            if not re.search(r"부식 상태.*(확인|점검)", full_text):
                result["부식_여부_patch"] = 1

    result["성능기록부_raw_patch"] = " || ".join(dict.fromkeys(evidence_lines)) if evidence_lines else None

    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if pd.isna(exchange_cnt) else int(exchange_cnt > 0)
    row["판금_여부"] = None if pd.isna(panel_cnt) else int(panel_cnt > 0)

    if pd.isna(exchange_cnt) and pd.isna(panel_cnt):
        row["외판수리_여부"] = None
        row["사고강도점수"] = None
    else:
        ex = 0 if pd.isna(exchange_cnt) else exchange_cnt
        pa = 0 if pd.isna(panel_cnt) else panel_cnt
        row["외판수리_여부"] = int((ex + pa) > 0)
        row["사고강도점수"] = ex * 2 + pa

    row["보험이력_여부"] = None if pd.isna(insurance_cnt) else int(insurance_cnt > 0)

    signals = [v for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]] if v is not None]
    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        ex = 0 if pd.isna(exchange_cnt) else exchange_cnt
        row["중대사고_추정"] = int(ex >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


# =========================
# 메인 보정
# =========================
df = pd.read_csv(RAW_CSV)

patch_rows = []

for idx, row in df.iterrows():
    brand = str(row["브랜드"])
    car_id = str(row["매물ID"])

    html_path = HTML_DIR / brand / car_id / "detail.html"

    if not html_path.exists():
        continue

    try:
        html = html_path.read_text(encoding="utf-8", errors="ignore")
        patch = parse_accident_from_html(html)
        patch["매물ID"] = car_id
        patch_rows.append(patch)
    except Exception as e:
        print(f"[ERROR] {brand} {car_id}: {e}")

patch_df = pd.DataFrame(patch_rows)

# 매물ID 기준 merge
df["매물ID"] = df["매물ID"].astype(str)
patch_df["매물ID"] = patch_df["매물ID"].astype(str)

df = df.merge(patch_df, on="매물ID", how="left")

# patch 값이 있으면 덮어쓰기
replace_map = {
    "보험이력건수_patch": "보험이력건수",
    "교환_개수_patch": "교환_개수",
    "판금_개수_patch": "판금_개수",
    "부식_여부_patch": "부식_여부",
    "성능기록부_raw_patch": "성능기록부_raw",
}

for patch_col, orig_col in replace_map.items():
    df[orig_col] = df[patch_col].combine_first(df[orig_col])

# patch 컬럼 삭제
df = df.drop(columns=list(replace_map.keys()))

# 사고 파생변수 재계산
df = df.apply(derive_accident_features, axis=1)

# 저장
df.to_csv(PATCHED_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {PATCHED_CSV}")
print(df[[
    "매물ID", "보험이력건수", "교환_개수", "판금_개수", "부식_여부",
    "사고강도점수", "사고종합_여부", "중대사고_추정"
]].head())

저장 완료: encar_domestic_hybrid_30000/output/domestic_raw_30000_accident_patched.csv
       매물ID  보험이력건수  교환_개수  판금_개수  부식_여부  사고강도점수  사고종합_여부  중대사고_추정
0  41187436     NaN    NaN    NaN    1.0     NaN      NaN      NaN
1  41383192     NaN    NaN    NaN    1.0     NaN      NaN      NaN
2  41635048     NaN    NaN    NaN    1.0     NaN      NaN      NaN
3  41524814     NaN    NaN    NaN    1.0     NaN      NaN      NaN
4  41485947     NaN    NaN    NaN    1.0     NaN      NaN      NaN


In [10]:
import requests
import json

url = "https://api.encar.com/v1/readside/inspection/vehicle/41587322"

headers = {
    "accept": "*/*",
    "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "origin": "https://fem.encar.com",
    "referer": "https://fem.encar.com/",
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
}

resp = requests.get(url, headers=headers, timeout=20)
print(resp.status_code)
print(resp.text[:3000])

200
{"vehicleId":41587322,"formats":["TABLE"],"master":{"supplyNum":"3040000657","accdient":true,"simpleRepair":true,"registrationDate":"2026-03-04T11:33:18","detail":{"recordNo":"3040000657","modelYear":"2012  ","validityStartDate":"20240726","validityEndDate":"20260725","firstRegistrationDate":"20120726","transmissionType":null,"vin":"KMHSW81XBDU","guarantyType":{"code":"2","title":"보험사보증"},"motorType":"D4HB","boardStateType":{"code":"1","title":"양호"},"mileage":139031,"mileageStateType":null,"coout":null,"hcout":null,"smout":null,"tuning":false,"tuningStateTypes":[],"seriousTypes":[],"usageChangeTypes":[],"colorType":null,"paintPartTypes":[],"mainOptionTypes":[],"recall":true,"recallFullFillTypes":[{"code":"1","title":"이행"}],"comments":"비금속(FRP 플라스틱)의 탈부착 가능 부품은 점검사항에서 제외되며 중고차 특성 상 부분적인 판금,도색 및 차량의 노후화에 따른 자연스러운 부식이 있을 수 있습니다. 기능상 영향없는 감각적 손해(소음/진동/냄새/외관/작동감각 등) 보증제외.","issueDate":"20260227","inspName":"한국자동차진단보증협회 제이피모빌리티(주) 이미연","noticeName":"친절한모터스","carStateType":{"code":"1","ti